In [5]:
import time
import tracemalloc
from dataclasses import dataclass

@dataclass
class RunStats:
    name: str
    n: int
    result: int
    elapsed_s: float
    peak_mem_bytes: int
    metrics: dict

# ----------------------------
# Fibonacci implementations
# ----------------------------

def fib_naive(n: int):
    """Naive recursion. Returns (result, metrics dict)."""
    calls = 0
    max_depth = 0

    def f(k: int, depth: int) -> int:
        nonlocal calls, max_depth
        calls += 1
        if depth > max_depth:
            max_depth = depth
        if k <= 1:
            return k
        return f(k - 1, depth + 1) + f(k - 2, depth + 1)

    result = f(n, 1)
    return result, {"calls": calls, "max_depth": max_depth}


def fib_dp(n: int):
    """Bottom-up DP with table. Returns (result, metrics dict)."""
    if n <= 1:
        return n, {"loop_iters": 0, "table_len": 0}

    F = [0] * (n + 1)
    F[0] = 0
    F[1] = 1

    loop_iters = 0
    for i in range(2, n + 1):
        F[i] = F[i - 1] + F[i - 2]
        loop_iters += 1

    return F[n], {"loop_iters": loop_iters, "table_len": len(F)}

# ----------------------------
# Measurement harness
# ----------------------------

def measure(fn, n: int, name: str) -> RunStats:
    tracemalloc.start()
    t0 = time.perf_counter()
    result, metrics = fn(n)
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return RunStats(
        name=name,
        n=n,
        result=result,
        elapsed_s=elapsed,
        peak_mem_bytes=peak,
        metrics=metrics,
    )

def show(stats: RunStats):
    print(f"{stats.name} (n={stats.n}) => {stats.result}")
    print(f"  time: {stats.elapsed_s:.6f}s")
    print(f"  peak mem: {stats.peak_mem_bytes:,} bytes")
    for k, v in stats.metrics.items():
        print(f"  {k}: {v}")

# ----------------------------
# Example run
# ----------------------------

s1 = measure(fib_naive, 20, "naive recursion")
s2 = measure(fib_dp, 20, "dynamic programming")

show(s1)
print()
show(s2)


naive recursion (n=20) => 6765
  time: 0.007522s
  peak mem: 9,433,203 bytes
  calls: 21891
  max_depth: 20

dynamic programming (n=20) => 6765
  time: 0.000021s
  peak mem: 440 bytes
  loop_iters: 19
  table_len: 21


In [4]:
print("hello")


hello
